[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanathi/Astro_Gateway_2026/blob/main/Notebooks/01_Gaia-Clusters-CMD.ipynb)

# 01 · แค็ตตาล็อกดาวและโฟโตเมตรี: ส่องกระจุกดาวลูกไก่ด้วยข้อมูล Gaia
**Astro Gateway** · คาบ 09.00–10.30 น.

**คำถามวันนี้:** กระจุกดาวลูกไก่ (Pleiades, M45) อยู่ห่างจากเราเท่าไร และดาวในกระจุกเป็นอย่างไร?

เราจะใช้ **Gaia DR3** — แค็ตตาล็อกจากยานอวกาศ Gaia ขององค์การอวกาศยุโรป (ESA) ที่วัดตำแหน่ง ระยะทาง
การเคลื่อนที่ และความสว่างของดาวเกือบ 2 พันล้านดวง

**ขั้นตอน:** ดึงข้อมูล → สำรวจ → แยกดาวในกระจุกออกจากดาวพื้นหลัง → หาระยะทาง → วาดแผนภาพสี-แมกนิจูด → เทียบกับกระจุกดาวอายุมาก

> ✏️ **ช่อง `____` ในโค้ดคือส่วนที่ต้องเติมเอง** (สูตรอยู่ในข้อความด้านบนหรือในสไลด์) — ถ้ารันแล้วเจอ `NameError: name '____' is not defined` แปลว่ายังเติมไม่ครบ

## 0. เตรียมเครื่องมือ
รันเซลล์ด้านล่าง (ฟังก์ชันช่วยอ่านไฟล์ข้อมูลสำรอง) แล้วติดตั้ง astroquery

In [ ]:
#@title 🔧 ฟังก์ชันช่วย — กดรันเซลล์นี้ก่อน (ไม่ต้องแก้ไข) { display-mode: "form" }
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
COURSE = "Astro_Gateway_2026"     # ชื่อโฟลเดอร์รายวิชา (GitHub: thanathi/Astro_Gateway_2026)
DATA_URL = "https://raw.githubusercontent.com/thanathi/Astro_Gateway_2026/main/Data/"   # ข้อมูลทั้งหมดบน GitHub · None = ใช้โฟลเดอร์ใน Google Drive แทน
DATA_DIRS = [".", "Data", "../Data", "/content/Data"]


def find_course(root="/content/drive"):
    """หาโฟลเดอร์ COURSE ใน Google Drive: ชั้นบนสุดของ MyDrive, ลึกลงไป 1-2 ชั้น หรือใน Shared drives
    เลือกโฟลเดอร์ที่มี Data อยู่ข้างใน (กันกรณีแตกไฟล์ zip แล้วได้โฟลเดอร์ซ้อนกัน 2 ชั้น)"""
    import glob, os
    hits = []
    for pat in ("MyDrive/{c}", "MyDrive/*/{c}", "MyDrive/*/*/{c}", "Shareddrives/*/{c}"):
        hits += sorted(glob.glob(f"{root}/{pat.format(c=COURSE)}"))
    good = [h for h in hits if os.path.isdir(os.path.join(h, "Data"))]
    return (good or hits or [None])[0]


def connect_drive():
    """ใน Colab: ถ้าใช้ข้อมูลจาก GitHub (DATA_URL) ไม่ต้องทำอะไร — ไฟล์จะดาวน์โหลดเองเมื่อใช้
    ถ้า DATA_URL = None: เชื่อม Google Drive แล้วย้ายไปที่โฟลเดอร์ Notebooks ของรายวิชา ให้ ../Data ใช้ได้
    นอก Colab ไม่ทำอะไร"""
    import os
    try:
        from google.colab import drive
    except ImportError:
        return
    if DATA_URL:
        print("ข้อมูลจะดาวน์โหลดจาก GitHub ให้อัตโนมัติเมื่อใช้ (ไม่ต้องเชื่อม Google Drive)")
        return
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    course = find_course()
    if course is None:
        print(f"⚠️ ไม่พบโฟลเดอร์ {COURSE} ใน Google Drive — โน้ตบุ๊กจะดาวน์โหลดข้อมูลจากคลังข้อมูลต้นทางแทน")
        print(f"   (คัดลอกโฟลเดอร์ {COURSE} ไว้ใน My Drive ตามวิธีใน README.txt แล้วรันเซลล์นี้ใหม่)")
        return
    nb = os.path.join(course, "Notebooks")
    os.chdir(nb if os.path.isdir(nb) else course)
    DATA_DIRS.insert(0, os.path.join(course, "Data"))
    print(f"เชื่อม Google Drive แล้ว: ใช้ข้อมูลจาก {os.path.join(course, 'Data')}")


def data_path(rel, url=None):
    """หาไฟล์ข้อมูล rel (เช่น "SDSS/spec-0531-52028-0573.fits"): ดูในโฟลเดอร์ Data ก่อน
    (หรือไฟล์ที่อัปโหลดในแถบ Files ของ Colab) ถ้าไม่มี ให้ดาวน์โหลดจาก url (คลังข้อมูลต้นทาง)"""
    import os
    name = os.path.basename(rel)
    for base in DATA_DIRS:
        for p in (os.path.join(base, rel), os.path.join(base, name)):
            if os.path.exists(p):
                return p
    for p in (os.path.join("/content", name),):                 # ไฟล์ที่ดาวน์โหลด/อัปโหลดไว้แล้วใน Colab
        if os.path.exists(p):
            return p
    # เขียนไฟล์ที่ดาวน์โหลดไว้ที่ /content ใน Colab (โฟลเดอร์ที่แชร์แบบดูอย่างเดียวเขียนไม่ได้)
    out = os.path.join("/content" if os.path.isdir("/content") else ".", name)
    if DATA_URL and not os.path.isdir("/content"):          # นอก Colab: ไม่ดาวน์โหลดทับโฟลเดอร์ทำงาน
        out = os.path.join(os.path.expanduser("~"), ".astro_gateway_cache", name)
        os.makedirs(os.path.dirname(out), exist_ok=True)
    for src in ([DATA_URL + rel] if DATA_URL else []) + ([url] if url else []):
        try:
            import requests
            r = requests.get(src, timeout=(10, 60))
            r.raise_for_status()
            with open(out, "wb") as fh:
                fh.write(r.content)
            print(f"ดาวน์โหลด {name} จาก {src.split('/')[2]} แล้ว ({len(r.content) / 1e6:.1f} MB)")
            return out
        except Exception as e:
            print(f"  ดาวน์โหลดจาก {src.split('/')[2]} ไม่ได้ ({type(e).__name__})")
    raise FileNotFoundError(f"ไม่พบไฟล์ {name} — ตรวจการเชื่อมต่ออินเทอร์เน็ต แล้วรันเซลล์นี้ใหม่ "
                            f"(หรือดาวน์โหลดไฟล์จาก github.com/thanathi/{COURSE} โฟลเดอร์ Data/{os.path.dirname(rel) or '.'} "
                            "แล้วอัปโหลดที่แถบ Files)")


# ไอโซโครน PARSEC v1.2S (Bressan et al. 2012) ในแถบ Gaia DR3 · log อายุ 6.6–10.0 ทุก 0.1
# ธาตุหนัก [M/H] = −2.0, −1.5, −1.0, −0.5, 0.0 (เท่าดวงอาทิตย์), +0.5
# ผลของฝุ่น (กฎ Cardelli, R_V = 3.1) ประมาณเป็นเส้นตรงตามสี คลาดเคลื่อน < 0.02 mag
_ISO = None


def age_text(log_age):
    """log อายุ → ข้อความภาษาอังกฤษสำหรับกราฟ เช่น 7.9 → '79 Myr', 9.6 → '4.0 Gyr'"""
    t = 10 ** log_age
    return f"{t / 1e6:,.0f} Myr" if t < 1e9 else f"{t / 1e9:.1f} Gyr"


def isochrone(log_age, av=0.0, mh=0.0):
    """ไอโซโครนที่อายุและธาตุหนักใกล้ค่าที่ให้มากที่สุด → (สี BP-RP, แมกนิจูดสัมบูรณ์ M_G, log อายุที่ใช้จริง)
    av = การดูดกลืนแสงจากฝุ่นในแถบ V (A_V): เส้นจะเลื่อนไปทางขวา (แดงขึ้น) และลง (จางลง)
    mh = ธาตุหนัก [M/H] (0 = เท่าดวงอาทิตย์, −1 = น้อยกว่า 10 เท่า)"""
    global _ISO
    if _ISO is None:
        _ISO = pd.read_csv(data_path("Isochrones/parsec_gaia_dr3.csv"))
    mhs = np.sort(_ISO.MH.unique())
    d = _ISO[_ISO.MH == mhs[np.argmin(np.abs(mhs - mh))]]
    ages = np.sort(d.logAge.unique())
    a = ages[np.argmin(np.abs(ages - log_age))]
    d = d[d.logAge == a]
    c0, lab = d.BP_RP.to_numpy(), d.label.to_numpy()
    c = c0 + av * (0.465 - 0.059 * c0)
    g = d.G.to_numpy() + av * (0.959 - 0.125 * c0)
    gap = np.flatnonzero((lab[1:] >= 4) != (lab[:-1] >= 4)) + 1      # ไม่ลากเส้นจากปลายดาวยักษ์แดงไปดาวเผาฮีเลียม
    return np.insert(c, gap, np.nan), np.insert(g, gap, np.nan), a


def age_slider(bp_rp, M_G, av=0.0, log_age=8.0, name="cluster", mh=0.0):
    """วาด CMD พร้อมแถบเลื่อนอายุ, A_V และ [M/H] — เลื่อนจนเส้นทับจุด turn-off และดาวยักษ์แดงพอดี"""
    import matplotlib.pyplot as plt

    def show(log_age=log_age, av=av, mh=mh):
        c, g, a = isochrone(log_age, av, mh)
        plt.figure(figsize=(6, 7))
        plt.scatter(bp_rp, M_G, s=5, color="tab:blue", label=name)
        plt.plot(c, g, color="tab:red", lw=2, label=f"isochrone {age_text(a)}, [M/H] = {mh:+.1f}")
        plt.xlim(-0.5, 3.4); plt.ylim(13, -4)
        plt.xlabel("colour BP - RP (mag)"); plt.ylabel("absolute magnitude M_G (mag)")
        plt.legend(markerscale=4); plt.title(f"{name}: log age = {a:.1f}, A_V = {av:.2f}"); plt.show()
    try:
        from ipywidgets import FloatSlider, interact
    except ImportError:
        return show()
    slider = lambda v, lo, hi, step, label: FloatSlider(value=v, min=lo, max=hi, step=step, description=label,
                                                        continuous_update=False)
    interact(show, log_age=slider(log_age, 6.6, 10.0, 0.1, "log age"), av=slider(av, 0.0, 2.5, 0.05, "A_V"),
             mh=slider(mh, -2.0, 0.5, 0.5, "[M/H]"))


connect_drive()
print("พร้อมใช้งาน ✓")

In [ ]:
# ติดตั้ง astroquery: ชุดเครื่องมือ Python สำหรับดึงข้อมูลจากคลังข้อมูลดาราศาสตร์
# (Colab ต้องติดตั้งใหม่ทุกครั้งที่เปิดโน้ตบุ๊ก ใช้เวลาไม่กี่วินาที · ข้อความ "restart the kernel" ไม่ต้องสนใจ)
%pip install -q --disable-pip-version-check astroquery

## 1. ดึงข้อมูลดาวจากแค็ตตาล็อก Gaia
แค็ตตาล็อก Gaia DR3 มีดาวเกือบ 2 พันล้านดวง — เราจะ "ถาม" ฐานข้อมูล **VizieR** (ศูนย์ข้อมูลดาราศาสตร์ Strasbourg ประเทศฝรั่งเศส)
ซึ่งเก็บแค็ตตาล็อกนี้ไว้ในชื่อ `I/355/gaiadr3` โดยขอเฉพาะดาวที่ตรงเงื่อนไข 3 ข้อ:
1. อยู่ในวงกลมรัศมี `radius` องศา รอบตำแหน่ง (`ra`, `dec`) ของกระจุก
2. สว่างกว่าแมกนิจูด `g_max` ในแถบ G
3. วัดพารัลแลกซ์ได้แม่น: พารัลแลกซ์มากกว่าความคลาดเคลื่อนอย่างน้อย `min_plx_over_err` เท่า

ฟังก์ชัน `get_gaia` ด้านล่างทำทั้งหมดนี้ — อ่านทีละบรรทัดแล้วรันเซลล์ (จะใช้ซ้ำกับกระจุกอื่นในข้อ 8)

In [ ]:
from astroquery.vizier import Vizier
from astropy.coordinates import SkyCoord
import astropy.units as u

# ชื่อคอลัมน์ใน VizieR -> ชื่อที่เราจะใช้ (ชื่อเดียวกับในฐานข้อมูลของ Gaia)
COLUMNS = {"Source": "source_id", "RA_ICRS": "ra", "DE_ICRS": "dec", "Plx": "parallax", "e_Plx": "parallax_error",
           "pmRA": "pmra", "pmDE": "pmdec", "Gmag": "phot_g_mean_mag", "BP-RP": "bp_rp", "RV": "radial_velocity"}

def get_gaia(ra, dec, radius, g_max=17, min_plx_over_err=10):
    """ดึงดาวจาก Gaia DR3 ในวงกลมรัศมี radius องศา รอบ (ra, dec) ผ่าน VizieR -> ตาราง pandas"""
    vizier = Vizier(columns=list(COLUMNS),                            # คอลัมน์ที่ต้องการ
                    column_filters={"Gmag": f"<{g_max}",              # เงื่อนไข: สว่างกว่า g_max
                                    "RPlx": f">{min_plx_over_err}"},  # เงื่อนไข: พารัลแลกซ์ / ความคลาดเคลื่อน
                    row_limit=-1, timeout=120)                        # -1 = เอาทุกดวง ไม่จำกัดจำนวน
    center = SkyCoord(ra, dec, unit="deg")                            # จุดกลางของวงกลมที่ค้น
    tables = vizier.query_region(center, radius=radius * u.deg, catalog="I/355/gaiadr3")
    df = tables[0].to_pandas().rename(columns=COLUMNS)
    return df.dropna(subset=["parallax", "pmra", "pmdec", "phot_g_mean_mag", "bp_rp"])   # ตัดดาวที่ข้อมูลไม่ครบ

In [ ]:
RA, DEC = 56.75, 24.12     # ตำแหน่งกลางของกระจุกดาวลูกไก่ (องศา)
RADIUS = 2.5               # รัศมีที่ค้น (องศา) กว้างประมาณ 10 เท่าของดวงจันทร์เต็มดวง

try:
    df = get_gaia(RA, DEC, RADIUS, g_max=17, min_plx_over_err=10)
    print(f"ดึงข้อมูลจาก VizieR ได้ {len(df):,} ดวง")
except Exception as e:                                   # เน็ตช้า/เซิร์ฟเวอร์ล่ม -> ใช้ไฟล์สำรอง
    print(f"ดึงข้อมูลไม่ได้ ({type(e).__name__}) — ใช้ไฟล์สำรองแทน")
    df = pd.read_csv(data_path("Gaia/pleiades_gaia_dr3.csv"))
df.head()

> ถ้าดึงข้อมูลไม่ได้ (เน็ตช้า/เซิร์ฟเวอร์ล่ม) โน้ตบุ๊กจะใช้ไฟล์สำรอง `Data/Gaia/pleiades_gaia_dr3.csv` ให้เอง

**ความหมายของคอลัมน์**
| คอลัมน์ | ความหมาย | หน่วย |
|---|---|---|
| `ra`, `dec` | ตำแหน่งบนท้องฟ้า | องศา |
| `parallax` | พารัลแลกซ์ (ยิ่งมาก ยิ่งใกล้) | mas (มิลลิอาร์กวินาที) |
| `pmra`, `pmdec` | การเคลื่อนที่เฉพาะ (ดาวเลื่อนบนท้องฟ้าปีละเท่าไร) | mas/ปี |
| `phot_g_mean_mag` | ความสว่างในแถบ G (ยิ่งน้อย ยิ่งสว่าง) | แมกนิจูด |
| `bp_rp` | สี = แมกนิจูดแถบน้ำเงิน − แถบแดง (ยิ่งน้อย ยิ่งร้อน) | แมกนิจูด |
| `radial_velocity` | ความเร็วเข้าหา/ออกจากเรา (มีเฉพาะบางดวง) | km/s |

In [ ]:
print(f"จำนวนดาว: {len(df):,} ดวง")
df[["parallax", "pmra", "pmdec", "phot_g_mean_mag", "bp_rp"]].describe().round(2)

✏️ **คำถาม:** ดาวที่สว่างที่สุดในข้อมูลมีแมกนิจูด G เท่าไร? (ใบ้: คอลัมน์ `phot_g_mean_mag` กับคำสั่ง `.min()` หรือ `.max()` — ดาวสว่าง = แมกนิจูดน้อย)

In [ ]:
g_brightest = ____
print(f"ดาวที่สว่างที่สุดมี G = {g_brightest:.2f}")

## 2. ดูตำแหน่งบนท้องฟ้า — เห็นกระจุกดาวไหม?

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(df.ra, df.dec, s=1, color="gray")
plt.gca().invert_xaxis()          # บนท้องฟ้า RA เพิ่มไปทางทิศตะวันออก (ซ้ายมือ)
plt.xlabel("RA (deg)"); plt.ylabel("Dec (deg)")
plt.title("Gaia stars around the Pleiades")
plt.show()

แทบมองไม่ออก! ดาวส่วนใหญ่ในภาพเป็น **ดาวพื้นหลัง** ที่อยู่ใกล้หรือไกลกว่ากระจุกมาก — เราต้องใช้ข้อมูลอื่นช่วยแยก

## 3. การเคลื่อนที่เฉพาะ (proper motion)
ดาวในกระจุกเกิดจากเมฆก๊าซก้อนเดียวกัน จึง **เคลื่อนที่ไปด้วยกัน** ในอวกาศ
ถ้าวาดกราฟ `pmra` กับ `pmdec` ดาวในกระจุกจะรวมเป็น **ก้อนเล็ก ๆ** แยกจากดาวพื้นหลัง

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(df.pmra, df.pmdec, s=1, color="gray")
plt.xlim(-30, 45); plt.ylim(-60, 25)
plt.xlabel("pmRA (mas/yr)"); plt.ylabel("pmDec (mas/yr)")
plt.title("Proper-motion diagram")
plt.grid(alpha=0.3)
plt.show()

✏️ **อ่านค่าจากกราฟ:** ก้อนเล็กที่แยกออกมาอยู่ที่ (pmRA, pmDec) ประมาณเท่าไร? ใส่ค่าลงในเซลล์ถัดไป

In [ ]:
PM_RA0, PM_DEC0 = ____    # ✏️ จุดกลางของก้อนที่อ่านได้จากกราฟ (pmRA, pmDec)
PM_RADIUS = 5                # รัศมีวงที่ใช้เลือก (mas/yr)

near_pm = np.hypot(df.pmra - PM_RA0, df.pmdec - PM_DEC0) < PM_RADIUS
print(f"ดาวที่เคลื่อนที่คล้ายกระจุก: {near_pm.sum()} ดวง")

## 4. พารัลแลกซ์ → ระยะทาง
ดาวในกระจุกอยู่ห่างจากเราเกือบเท่ากัน จึงควรมีพารัลแลกซ์ใกล้เคียงกัน

In [ ]:
bins = np.arange(0, 12, 0.1)
plt.figure(figsize=(8, 3.5))
plt.hist(df.parallax, bins=bins, color="lightgray", label="all stars")
plt.hist(df.parallax[near_pm], bins=bins, color="tab:orange", label="similar proper motion")
plt.xlabel("parallax (mas)"); plt.ylabel("number of stars")
plt.legend(); plt.show()

✏️ ยอดสีส้มอยู่ช่วงพารัลแลกซ์เท่าไร? ใส่ขอบเขตในเซลล์ถัดไป

In [ ]:
PLX_MIN, PLX_MAX = ____          # ✏️ ช่วงพารัลแลกซ์ของกระจุก (mas) อ่านจากฮิสโทแกรม

good_plx = df.parallax.between(PLX_MIN, PLX_MAX)
is_member = near_pm & good_plx
members, field = df[is_member], df[~is_member]
print(f"สมาชิกกระจุก: {len(members)} ดวง จากทั้งหมด {len(df):,} ดวง")

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(field.ra, field.dec, s=1, color="lightgray", label="field")
plt.scatter(members.ra, members.dec, s=4, color="tab:orange", label="members")
plt.gca().invert_xaxis()
plt.xlabel("RA (deg)"); plt.ylabel("Dec (deg)")
plt.legend(markerscale=4); plt.title("Pleiades members"); plt.show()

## 5. กระจุกดาวลูกไก่อยู่ไกลแค่ไหน?
**ระยะทาง (พาร์เซก) = 1000 / พารัลแลกซ์ (mas)** — 1 พาร์เซก ≈ 3.26 ปีแสง
เราใช้ **ค่ามัธยฐาน (median)** ของสมาชิกทุกดวง เพราะทนต่อค่าผิดปกติได้ดีกว่าค่าเฉลี่ย

In [ ]:
plx_med = members.parallax.median()
plx_err = 1.25 * members.parallax.std() / np.sqrt(len(members))   # ความคลาดเคลื่อนโดยประมาณของค่ามัธยฐาน
dist = ____                                       # ✏️ ระยะทาง (พาร์เซก)
dist_err = dist * plx_err / plx_med

print(f"พารัลแลกซ์ = {plx_med:.3f} ± {plx_err:.3f} mas")
print(f"ระยะทาง   = {dist:.1f} ± {dist_err:.1f} พาร์เซก ≈ {dist * 3.26:.0f} ปีแสง")

งานวิจัยที่ใช้ข้อมูล Gaia ได้ระยะทางของกระจุกดาวลูกไก่ประมาณ **136 พาร์เซก** — ได้ใกล้เคียงไหม?

🔎 **คิดต่อ:** ทำไมการใช้ดาวหลายร้อยดวงจึงแม่นกว่าการวัดดาวดวงเดียว?

## 6. แผนภาพสี-แมกนิจูด (Colour–Magnitude Diagram, CMD)
แกน x = **สี** (BP − RP): ซ้าย = น้ำเงิน/ร้อน, ขวา = แดง/เย็น
แกน y = **แมกนิจูด**: ต้อง **กลับแกน** ให้ดาวสว่างอยู่ด้านบน

In [ ]:
plt.figure(figsize=(6, 7))
plt.scatter(field.bp_rp, field.phot_g_mean_mag, s=1, color="lightgray", label="field")
plt.scatter(members.bp_rp, members.phot_g_mean_mag, s=5, color="tab:orange", label="Pleiades")
# ✏️ เพิ่มคำสั่งกลับแกน y ตรงนี้ ให้ดาวสว่างอยู่ด้านบน (ดูตารางสรุปท้ายโน้ตบุ๊ก 00)
plt.xlim(-0.5, 3.6)
plt.xlabel("colour BP - RP (mag)"); plt.ylabel("G (mag)")
plt.legend(markerscale=4); plt.title("Colour-magnitude diagram"); plt.show()

สมาชิกเรียงตัวเป็นแถบโค้งชัดเจน — นี่คือ **แถบลำดับหลัก (main sequence)**: ดาวที่กำลังเผาไหม้ไฮโดรเจนที่แกนกลาง

## 7. แมกนิจูดสัมบูรณ์: ความสว่าง "จริง" ของดาว
แมกนิจูดสัมบูรณ์ คือแมกนิจูดที่จะเห็นถ้าดาวอยู่ห่าง 10 พาร์เซก

**M = m + 5 log₁₀(พารัลแลกซ์ [mas] / 100)**

In [ ]:
members = members.assign(M_G=____)   # ✏️ M = m + 5 log₁₀(พารัลแลกซ์ / 100)

plt.figure(figsize=(6, 7))
plt.scatter(members.bp_rp, members.M_G, s=5, color="tab:orange", label="Pleiades")
plt.plot(0.82, 4.67, "*", ms=18, color="gold", mec="k", label="the Sun")   # ดวงอาทิตย์
plt.gca().invert_yaxis()
plt.xlabel("colour BP - RP (mag)"); plt.ylabel("absolute magnitude M_G (mag)")
plt.legend(); plt.title("Pleiades in absolute magnitude"); plt.show()

🔎 **คำถาม:** ดวงอาทิตย์อยู่ตรงไหนเมื่อเทียบกับดาวในกระจุกดาวลูกไก่? ดาวที่ร้อนกว่าดวงอาทิตย์สว่างกว่าหรือจางกว่า?

## 7.1 หาอายุกระจุกดาวด้วยไอโซโครน (isochrone)
**ไอโซโครน** = เส้นที่แบบจำลองวิวัฒนาการดาวฤกษ์คำนวณว่า ดาวที่ **เกิดพร้อมกัน** แต่มวลต่างกัน จะอยู่ตรงไหนบน CMD เมื่อเวลาผ่านไป
- ดาวมวลมากใช้เชื้อเพลิงหมดก่อน → ปลายบนของแถบลำดับหลัก "หักเลี้ยว" ไปทางขวา (**turn-off**) ยิ่งกระจุกแก่ จุดนี้ยิ่งต่ำลงและแดงขึ้น
- ปลายล่างของแถบลำดับหลักแทบไม่เปลี่ยนตามอายุ → **อายุดูจากจุด turn-off และดาวยักษ์แดงเป็นหลัก**
- ฝุ่นระหว่างดาวทำให้ดาวดูจางลงและแดงขึ้น ใส่ค่า **A_V** (จากตาราง) เพื่อเลื่อนเส้นให้ตรงกับข้อมูล

ไอโซโครนในโน้ตบุ๊กนี้มาจากแบบจำลอง **PARSEC** (Bressan et al. 2012) — ค่าเริ่มต้นคือดาวที่มีธาตุหนักเท่าดวงอาทิตย์ ([M/H] = 0)

อายุเขียนเป็น **log₁₀(อายุเป็นปี)** เช่น log อายุ = 8 หมายถึง 10⁸ ปี = 100 ล้านปี

In [ ]:
plt.figure(figsize=(6, 7))
plt.scatter(members.bp_rp, members.M_G, s=5, color="tab:orange", label="Pleiades")
for log_age, colour in [(7.0, "tab:blue"), (8.0, "tab:red"), (9.0, "tab:green")]:   # 10 ล้าน, 100 ล้าน, 1,000 ล้านปี
    c, g, a = isochrone(log_age, av=0.18)          # A_V ของกระจุกดาวลูกไก่ = 0.18 mag
    plt.plot(c, g, color=colour, lw=1.5, label=f"isochrone {age_text(a)}")
plt.xlim(-0.5, 3.4); plt.ylim(13, -4)
plt.xlabel("colour BP - RP (mag)"); plt.ylabel("absolute magnitude M_G (mag)")
plt.legend(markerscale=4); plt.title("Pleiades with isochrones"); plt.show()

🔎 **คำถาม:** เส้นไหนเข้ากับดาวสีน้ำเงินที่สว่างที่สุดของกระจุกดาวลูกไก่? ทำไมทั้งสามเส้นจึงทับกันที่ปลายล่าง (ดาวแดง จาง)?

ลอง **เลื่อนแถบ log age** ด้านล่างจนเส้นสีแดงผ่านดาวสีน้ำเงินที่สว่างที่สุดพอดี (ปล่อยเมาส์แล้วกราฟจะวาดใหม่)

In [ ]:
age_slider(members.bp_rp, members.M_G, av=0.18, log_age=8.0, name="Pleiades")

In [ ]:
PLEIADES_LOG_AGE = ____                     # ✏️ log อายุที่เส้นเข้ากับข้อมูลที่สุด (อ่านจากแถบเลื่อน)
age_myr = ____     # ✏️ แปลงเป็นล้านปี: อายุ = 10^(log อายุ) ปี
print(f"กระจุกดาวลูกไก่อายุประมาณ {age_myr:,.0f} ล้านปี")

> 💡 กระจุกดาวลูกไก่มีดาวสว่างที่ turn-off อยู่ไม่กี่ดวง อายุจึงบอกได้แค่คร่าว ๆ (±0.1–0.2 ใน log อายุ)
> งานวิจัยใช้วิธีอื่นช่วยด้วย เช่น ปริมาณลิเทียมในดาวมวลน้อย ได้อายุประมาณ 80–130 ล้านปี
> กระจุกที่แก่กว่า (เช่น M67) มี turn-off และดาวยักษ์แดงชัดเจน จึงหาอายุได้แม่นกว่ามาก — ลองในแบบฝึกหัดถัดไป

## 8. แบบฝึกหัด: เลือกกระจุกดาวของตัวเอง
ทำขั้นตอนเดิมกับกระจุกดาวอีกแห่ง แล้วเปรียบเทียบกับกระจุกดาวลูกไก่ — ถ้ายังไม่รู้จะเลือกอะไร เริ่มจาก **M67** (กระจุกอายุมาก)
1. เลือกกระจุกจากตาราง แล้วดึงข้อมูลด้วย `get_gaia`
2. ดูแผนภาพ proper motion แล้วหาจุดกลางของก้อน
3. ดูฮิสโทแกรมพารัลแลกซ์แล้วกำหนดช่วง
4. คำนวณระยะทาง แล้ววาด CMD แมกนิจูดสัมบูรณ์เทียบกับกระจุกดาวลูกไก่
5. หาอายุด้วยไอโซโครน (ใช้ค่า A_V จากตาราง)

| กระจุก | ชื่อในแค็ตตาล็อก | RA (°) | Dec (°) | ระยะทาง (pc) | ฝุ่น A_V (mag) | อายุ (ปี) |
|---|---|---|---|---|---|---|
| h Persei | NGC 869 | 34.74 | +57.13 | 2,197 | 1.69 | 15 ล้าน |
| IC 2391 | IC 2391 | 130.29 | -52.99 | 148 | 0.04 | 29 ล้าน |
| NGC 2547 | NGC 2547 | 122.53 | -49.20 | 396 | 0.14 | 32 ล้าน |
| IC 2602 | IC 2602 | 160.61 | -64.43 | 149 | 0.03 | 36 ล้าน |
| alpha Persei | Melotte 20 | 51.62 | +48.98 | 171 | 0.30 | 51 ล้าน |
| Pleiades | Melotte 22 | 56.60 | +24.11 | 128 | 0.18 | 78 ล้าน |
| Blanco 1 | Blanco 1 | 0.85 | -29.96 | 240 | 0.01 | 105 ล้าน |
| M47 | NGC 2422 | 114.15 | -14.49 | 504 | 0.18 | 110 ล้าน |
| M34 | NGC 1039 | 40.53 | +42.72 | 534 | 0.16 | 132 ล้าน |
| M35 | NGC 2168 | 92.27 | +24.34 | 906 | 0.46 | 148 ล้าน |
| M41 | NGC 2287 | 101.50 | -20.72 | 688 | 0.02 | 170 ล้าน |
| NGC 2516 | NGC 2516 | 119.53 | -60.80 | 423 | 0.11 | 240 ล้าน |
| M11 | NGC 6705 | 282.77 | -6.27 | 2,203 | 1.20 | 309 ล้าน |
| M48 | NGC 2548 | 123.41 | -5.73 | 772 | 0.15 | 389 ล้าน |
| NGC 3532 | NGC 3532 | 166.42 | -58.71 | 498 | 0.00 | 398 ล้าน |
| M37 | NGC 2099 | 88.07 | +32.55 | 1,432 | 0.75 | 447 ล้าน |
| Praesepe | NGC 2632 | 130.05 | +19.62 | 183 | 0.00 | 676 ล้าน |
| NGC 6633 | NGC 6633 | 276.85 | +6.62 | 424 | 0.30 | 692 ล้าน |
| NGC 2360 | NGC 2360 | 109.44 | -15.63 | 1,122 | 0.39 | 1.0 พันล้าน |
| NGC 752 | NGC 752 | 29.22 | +37.79 | 483 | 0.07 | 1.2 พันล้าน |
| NGC 7789 | NGC 7789 | 359.33 | +56.73 | 2,100 | 0.83 | 1.5 พันล้าน |
| IC 4651 | IC 4651 | 261.21 | -49.92 | 982 | 0.42 | 1.7 พันล้าน |
| NGC 6819 | NGC 6819 | 295.33 | +40.19 | 2,765 | 0.40 | 2.2 พันล้าน |
| Ruprecht 147 | Ruprecht 147 | 289.09 | -16.33 | 323 | 0.06 | 3.0 พันล้าน |
| M67 | NGC 2682 | 132.85 | +11.81 | 889 | 0.07 | 4.3 พันล้าน |
| NGC 6791 | NGC 6791 | 290.22 | +37.78 | 4,231 | 0.70 | 6.3 พันล้าน |
| NGC 188 | NGC 188 | 11.80 | +85.24 | 1,698 | 0.21 | 7.1 พันล้าน |

> 💡 ระยะทางและอายุในตารางมาจาก Cantat-Gaudin et al. (2020) — ใช้ตรวจคำตอบหลังทำเสร็จ
> กระจุกที่อยู่ไกล (เกิน ~1,500 pc) ก้อนใน proper motion จะเล็กมาก (< 1 mas/ปี) ต้องซูมกราฟ ·
> กระจุกที่อยู่ในแนวทางช้างเผือก (เช่น IC 2391, IC 2602, NGC 3532) มีดาวพื้นหลังมาก อาจใช้เวลาดึงข้อมูล 20–60 วินาที

In [ ]:
CLUSTERS = {   # ชื่อ: (RA, Dec, รัศมีที่ค้น [°], g_max, min_plx_over_err) — ตั้งไว้ให้ได้ดาวไม่มากจนดึงข้อมูลช้า
    "h Persei":      (  34.741,   57.134, 0.30, 18,  3),
    "IC 2391":       ( 130.292,  -52.991, 1.30, 17, 10),
    "NGC 2547":      ( 122.525,  -49.198, 0.45, 17, 10),
    "IC 2602":       ( 160.613,  -64.426, 1.20, 16, 10),
    "alpha Persei":  (  51.617,   48.975, 2.00, 17, 10),
    "Pleiades":      (  56.601,   24.114, 2.50, 17, 10),
    "Blanco 1":      (   0.853,  -29.958, 1.12, 17, 10),
    "M47":           ( 114.147,  -14.489, 0.41, 17, 10),
    "M34":           (  40.531,   42.722, 0.64, 17,  5),
    "M35":           (  92.272,   24.336, 0.51, 18,  5),
    "M41":           ( 101.499,  -20.716, 0.53, 18,  5),
    "NGC 2516":      ( 119.527,  -60.800, 1.00, 17, 10),
    "M11":           ( 282.766,   -6.272, 0.30, 18,  3),
    "M48":           ( 123.412,   -5.726, 0.59, 18,  5),
    "NGC 3532":      ( 166.417,  -58.707, 0.86, 17, 10),
    "M37":           (  88.074,   32.545, 0.30, 18,  3),
    "Praesepe":      ( 130.054,   19.621, 1.50, 17, 10),
    "NGC 6633":      ( 276.845,    6.615, 0.58, 17, 10),
    "NGC 2360":      ( 109.443,  -15.631, 0.30, 18,  5),
    "NGC 752":       (  29.223,   37.794, 0.78, 17, 10),
    "NGC 7789":      ( 359.334,   56.726, 0.34, 18,  3),
    "IC 4651":       ( 261.212,  -49.917, 0.37, 18,  5),
    "NGC 6819":      ( 295.327,   40.190, 0.30, 18,  3),
    "Ruprecht 147":  ( 289.087,  -16.333, 1.04, 17, 10),
    "M67":           ( 132.846,   11.814, 0.60, 18,  5),
    "NGC 6791":      ( 290.221,   37.778, 0.30, 18,  3),
    "NGC 188":       (  11.798,   85.244, 0.50, 18,  3),
}
BACKUP = {"Pleiades": "Gaia/pleiades_gaia_dr3.csv", "NGC 2516": "Gaia/ngc2516_gaia_dr3.csv",    # ไฟล์สำรอง (มีแค่ 5 กระจุก)
          "Praesepe": "Gaia/praesepe_gaia_dr3.csv", "M67": "Gaia/m67_gaia_dr3.csv", "NGC 188": "Gaia/ngc188_gaia_dr3.csv"}

NAME = "M67"                       # 🔧 เลือกกระจุกจากตาราง (พิมพ์ให้ตรงกับชื่อใน CLUSTERS)
ra, dec, radius, g_max, poe = CLUSTERS[NAME]
try:
    cl = get_gaia(ra, dec, radius, g_max=g_max, min_plx_over_err=poe)
    print(f"{NAME}: ดึงข้อมูลได้ {len(cl):,} ดวง")
except Exception as e:
    print(f"ดึงข้อมูลไม่ได้ ({type(e).__name__}) — ใช้ไฟล์สำรอง (ถ้ามี)")
    cl = pd.read_csv(data_path(BACKUP[NAME]))

In [ ]:
PM_ZOOM = None      # 🔧 ซูมกราฟ proper motion: ใส่ (x_min, x_max, y_min, y_max) เช่น (-15, -7, -7, 1) แล้วรันเซลล์นี้ใหม่

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(cl.pmra, cl.pmdec, s=1, color="gray")
x0, x1 = np.percentile(cl.pmra, [0.5, 99.5]); y0, y1 = np.percentile(cl.pmdec, [0.5, 99.5])
ax[0].set(xlim=PM_ZOOM[:2] if PM_ZOOM else (x0 - 2, x1 + 2), ylim=PM_ZOOM[2:] if PM_ZOOM else (y0 - 2, y1 + 2),
          xlabel="pmRA (mas/yr)", ylabel="pmDec (mas/yr)", title=f"{NAME}: proper motions")
ax[0].grid(alpha=0.3)
ax[1].hist(cl.parallax, bins=np.arange(-0.5, cl.parallax.quantile(0.995) + 0.5, 0.05), color="gray")
ax[1].set(xlabel="parallax (mas)", ylabel="number", title=f"{NAME}: parallaxes")
plt.show()

In [ ]:
# ✏️ เติมค่าที่อ่านได้จากกราฟด้านบน
CL_PM0 = ____          # จุดกลางของก้อน (pmRA, pmDec)
CL_PM_RADIUS = 1.0                  # 🔧 รัศมีวงที่ใช้เลือก (mas/ปี): กระจุกใกล้ใช้ 2–5, กระจุกไกลใช้ 0.5–1
CL_PLX = ____              # ช่วงพารัลแลกซ์ (ต่ำสุด, สูงสุด) mas

sel = (np.hypot(cl.pmra - CL_PM0[0], cl.pmdec - CL_PM0[1]) < CL_PM_RADIUS) & cl.parallax.between(*CL_PLX)
cl_mem = cl[sel].assign(M_G=lambda d: d.phot_g_mean_mag + 5 * np.log10(d.parallax / 100))
print(f"สมาชิก {NAME}: {len(cl_mem)} ดวง | ระยะทาง ≈ {1000 / cl_mem.parallax.median():.0f} พาร์เซก")

In [ ]:
plt.figure(figsize=(6, 7))
plt.scatter(cl_mem.bp_rp, cl_mem.M_G, s=5, color="tab:blue", label=NAME)
plt.scatter(members.bp_rp, members.M_G, s=5, color="tab:orange", label="Pleiades")
plt.gca().invert_yaxis()
plt.xlim(-0.5, 3.4); plt.ylim(13, -3)
plt.xlabel("colour BP - RP (mag)"); plt.ylabel("absolute magnitude M_G (mag)")
plt.legend(markerscale=4); plt.title(f"Pleiades vs {NAME}"); plt.show()

### หาอายุของกระจุกที่เลือก
เลื่อนแถบ **log age** จนเส้นผ่าน **จุด turn-off** และ **ดาวยักษ์แดง** (ดาวที่แยกขึ้นไปทางขวาบน) — ถ้าเส้นเลื่อนซ้าย/ขวาไม่ตรง ลองปรับ **A_V** เล็กน้อย
> 🔎 (เลือกทำ) แถบ **[M/H]** เปลี่ยนปริมาณธาตุหนัก — ลองดูว่าเส้นเปลี่ยนไปทางไหน แล้วเปรียบเทียบกับผลของ A_V: แยกสองอย่างนี้ออกจากกันได้ไหม?
> 💡 ดาวที่อยู่ **เหนือ** turn-off สีน้ำเงิน (blue stragglers) เกิดจากดาวคู่ที่ถ่ายเทมวลหรือชนกัน — ไม่ต้องให้เส้นผ่านดาวพวกนี้

In [ ]:
A_V = {   # การดูดกลืนแสงจากฝุ่น (mag) จาก Cantat-Gaudin et al. (2020)
    "h Persei": 1.69, "IC 2391": 0.04, "NGC 2547": 0.14, "IC 2602": 0.03, "alpha Persei": 0.30, "Pleiades": 0.18,
    "Blanco 1": 0.01, "M47": 0.18, "M34": 0.16, "M35": 0.46, "M41": 0.02, "NGC 2516": 0.11,
    "M11": 1.20, "M48": 0.15, "NGC 3532": 0.00, "M37": 0.75, "Praesepe": 0.00, "NGC 6633": 0.30,
    "NGC 2360": 0.39, "NGC 752": 0.07, "NGC 7789": 0.83, "IC 4651": 0.42, "NGC 6819": 0.40, "Ruprecht 147": 0.06,
    "M67": 0.07, "NGC 6791": 0.70, "NGC 188": 0.21,
}
age_slider(cl_mem.bp_rp, cl_mem.M_G, av=A_V.get(NAME, 0.0), log_age=8.0, name=NAME)

In [ ]:
CL_LOG_AGE = ____                           # ✏️ log อายุที่เส้นเข้ากับข้อมูลที่สุด
print(f"{NAME} อายุประมาณ {10 ** CL_LOG_AGE / 1e6:,.0f} ล้านปี = {10 ** CL_LOG_AGE / 1e9:.1f} พันล้านปี")
print(f"แก่กว่ากระจุกดาวลูกไก่ประมาณ {10 ** (CL_LOG_AGE - PLEIADES_LOG_AGE):.0f} เท่า")

### (เลือกทำ) อยากได้กระจุกอื่นนอกตาราง? ค้นจากแค็ตตาล็อกกระจุกดาวกว่า 2,000 แห่ง
แค็ตตาล็อกของ Cantat-Gaudin et al. (2020) ที่สร้างจากข้อมูล Gaia — ชื่อใช้ `_` แทนช่องว่าง เช่น `NGC_2682` (M67), `Melotte_22` (ดาวลูกไก่)
ได้ค่า RA, Dec, รัศมี `r50` (องศา, มีสมาชิกครึ่งหนึ่งอยู่ข้างใน) แล้วเพิ่มลงใน `CLUSTERS` ด้วยรัศมีที่ค้นประมาณ 1.5 × r50

In [ ]:
CG_NAME = "NGC_2682"        # 🔧 ชื่อกระจุกในแค็ตตาล็อก
cg = Vizier(columns=["Cluster", "RA_ICRS", "DE_ICRS", "r50", "pmRA*", "pmDE", "plx", "DistPc", "AgeNN", "AVNN"], row_limit=-1)
info = cg.query_constraints(catalog="J/A+A/640/A1/table1", Cluster=CG_NAME)
print("AgeNN = log₁₀(อายุเป็นปี) · AVNN = ฝุ่น A_V (mag) · DistPc = ระยะทาง (พาร์เซก) · pmRA_, pmDE, plx = ค่าเฉลี่ยของสมาชิก")
info[0].to_pandas() if len(info) else print("ไม่พบชื่อนี้ — ตรวจตัวสะกดและใช้ _ แทนช่องว่าง")

🔎 **คำถามชวนคิด**
1. กระจุกที่เลือกมีดาวสีน้ำเงินที่สว่างมาก ๆ เหมือนกระจุกดาวลูกไก่ไหม? จุดที่แถบลำดับหลัก "หักเลี้ยว" (turn-off) อยู่ตรงไหน?
2. ดาวมวลมากใช้เชื้อเพลิงเร็วกว่าดาวมวลน้อย — แล้วกระจุกไหน **อายุมากกว่า**? อายุจากไอโซโครนตรงกับอายุในตารางไหม? ถ้าไม่ตรง เพราะอะไร?
3. ถ้าเห็นดาวกลุ่มที่แยกขึ้นไปด้านขวาบน (สว่างและแดง) คือดาวอะไร?
4. กระจุกที่อยู่ไกลมักดูแดงและจางกว่าที่ควร เพราะ **ฝุ่น** ระหว่างดาวบังแสง — กระจุกของคุณเป็นแบบนั้นไหม?
5. รวมผลกับเพื่อนทั้งห้อง: เรียงกระจุกตามอายุที่วัดได้ — กระจุกที่แก่ที่สุดในห้องอายุเท่าไร เทียบกับอายุเอกภพ (13,800 ล้านปี)?

### ✅ เช็กลิสต์การนำเสนอข้อมูลด้วยแผนภาพ
- แกนทุกแกนมีชื่อ **และหน่วย**
- กลับแกนแมกนิจูดให้ดาวสว่างอยู่ด้านบน
- ใช้สีเทาสำหรับข้อมูลพื้นหลัง และสีเด่นสำหรับสิ่งที่ต้องการให้ดู
- มีคำอธิบายสัญลักษณ์ (legend) เมื่อมีข้อมูลมากกว่าหนึ่งชุด
- จุดเล็กพอที่จะไม่ทับกันจนมองไม่เห็นโครงสร้าง